In [75]:

import pandas as pd

# Part 1. Repository Mining

In [76]:
# Load the columns of interest from the pull request dataset
selected_columns = [
    "id", "number",  "title", "body","agent", "user", "state","created_at", "closed_at", "merged_at", "repo_url"]

pull_request_df = pd.read_parquet("hf://datasets/hao-li/AIDev@v3/pull_request.parquet", columns=selected_columns)

pr_reviews_df = pd.read_parquet("hf://datasets/hao-li/AIDev@v3/pr_reviews.parquet")
pr_comments_df = pd.read_parquet("hf://datasets/hao-li/AIDev@v3/pr_comments.parquet")
pr_commits_df = pd.read_parquet("hf://datasets/hao-li/AIDev@v3/pr_commits.parquet")

print(pull_request_df.columns.tolist())


print(f"Number of PRs: {len(pull_request_df)}")

# Print the columns to verify the join
print(pr_reviews_df.columns.tolist())
print(pr_comments_df.columns.tolist())
print(pr_commits_df.columns.tolist())

['id', 'number', 'title', 'body', 'agent', 'user', 'state', 'created_at', 'closed_at', 'merged_at', 'repo_url']
Number of PRs: 33596
['id', 'pr_id', 'user', 'user_type', 'state', 'submitted_at', 'body']
['id', 'pr_id', 'user', 'user_id', 'user_type', 'created_at', 'body']
['sha', 'pr_id', 'author', 'committer', 'message']


In [77]:
# Load the columns for RQ1
pr_task_type_df = pd.read_parquet("hf://datasets/hao-li/AIDev@v3/pr_task_type.parquet", columns=["id", "type", "confidence"])

pr_commit_details_df = pd.read_parquet("hf://datasets/hao-li/AIDev@v3/pr_commit_details.parquet", 
    columns=["pr_id", "commit_stats_additions", "commit_stats_deletions", "filename"])

pr_timeline_df = pd.read_parquet("hf://datasets/hao-li/AIDev@v3/pr_timeline.parquet",
    columns=["pr_id", "event", "created_at"])

# Part 2. Data Cleaning & Classification

In [78]:
# Remove duplicates
pr_counter = len(pull_request_df)
pull_request_df = pull_request_df.drop_duplicates(subset=['id'])
print(f"Number of PRs after removing duplicates: {len(pull_request_df)}")

Number of PRs after removing duplicates: 33596


In [79]:
#  Handle missing values
pull_request_df["is_merged"] = pull_request_df["merged_at"].notna()
pull_request_df["is_closed"] = pull_request_df["closed_at"].notna()

pull_request_df["has_body"] = pull_request_df["body"].notna() & (pull_request_df["body"].str.strip() != "")

print(pull_request_df.isna().sum())

id               0
number           0
title            0
body           360
agent            0
user             0
state            0
created_at       0
closed_at     2312
merged_at     9582
repo_url         0
is_merged        0
is_closed        0
has_body         0
dtype: int64


In [80]:
# # Identify AI-related vs human

In [81]:
# Classify task types 
pr_task_type_df = pr_task_type_df.drop_duplicates(subset=['id'])

pull_request_df = pull_request_df.merge(
    pr_task_type_df[["id", "type", "confidence"]],
    on="id",
    how="left"
)
pull_request_df["type"] = pull_request_df["type"].fillna("unclassified")
print(pull_request_df["type"].value_counts(dropna=False))

type
feat        14450
fix          8106
docs         3887
test         2356
refactor     2288
chore         896
build         627
ci            411
perf          340
style         188
other          31
revert         16
Name: count, dtype: int64


In [82]:
# Print the final dataset 
pull_request_df.head()

,id,number,title,body,agent,user,state,created_at,closed_at,merged_at,repo_url,is_merged,is_closed,has_body,type,confidence
0,3264933329,2911,Fix: Wait for all partitions in load_collectio...,## Summary\n\nFixes an issue where `load_colle...,Claude_Code,weiliu1031,closed,2025-07-26T02:59:01Z,2025-07-29T07:01:20Z,NaN,https://api.github.com/repos/milvus-io/pymilvus,False,True,True,fix,10
1,3265118634,2,ファイルパス参照を相対パスに統一し、doc/からdocs/に統一,## 背景\n\n現在、本プロジェクトにおいて以下のパス構成の不整合が生じています：\n\n...,Claude_Code,cm-kojimat,closed,2025-07-26T04:56:55Z,2025-07-26T22:12:24Z,2025-07-26T22:12:24Z,https://api.github.com/repos/classmethod/tsumiki,True,True,True,refactor,9
2,3265640341,30,Add build staleness detection for debug CLI,## Summary\r\n\r\n Implements comprehensive b...,Claude_Code,MSch,closed,2025-07-26T13:31:19Z,2025-07-26T13:37:22Z,2025-07-26T13:37:22Z,https://api.github.com/repos/steipete/Peekaboo,True,True,True,feat,10
3,3265709660,205,feat: add comprehensive README screenshots wit...,## Type of Change\n\n- [ ] 🐛 `bug` - Bug fix (...,Claude_Code,sugyan,closed,2025-07-26T14:07:22Z,2025-07-26T14:45:30Z,2025-07-26T14:45:30Z,https://api.github.com/repos/sugyan/claude-cod...,True,True,True,feat,10
4,3265782173,17625,chore: remove HashedPostStateProvider trait,## Summary\r\n\r\n#17545 \r\n\r\nRemove the un...,Claude_Code,adust09,open,2025-07-26T15:02:48Z,NaN,NaN,https://api.github.com/repos/paradigmxyz/reth,False,False,True,chore,10


# Part 3. Metrics Definition

In [83]:
for col in ["created_at", "closed_at", "merged_at"]:
    pull_request_df[col] = pd.to_datetime(pull_request_df[col], utc=True)


# Merge rate
merge_rate = pull_request_df["is_merged"].mean()
print("Merge rate:", merge_rate)

# Time to merge (h)
pull_request_df["time_to_merge_hours"] = (
    (pull_request_df["merged_at"] - pull_request_df["created_at"]).dt.total_seconds() / 3600
)
print("Time to merge (h), median:", pull_request_df["time_to_merge_hours"].median())

# 3. Time to close (h)
pull_request_df["time_to_close_hours"] = (
    (pull_request_df["closed_at"] - pull_request_df["created_at"]).dt.total_seconds() / 3600
)
print("Time to close (h), median:", pull_request_df["time_to_close_hours"].median())

# Number of review rounds
review_rounds = pr_reviews_df.groupby("pr_id").size().rename("review_rounds")
pull_request_df = pull_request_df.merge(review_rounds, left_on="id", right_index=True, how="left")
pull_request_df["review_rounds"] = pull_request_df["review_rounds"].fillna(0)
print("Review rounds, median:", pull_request_df["review_rounds"].median())


Merge rate: 0.714787474699369
Time to merge (h), median: 0.04555555555555556
Time to close (h), median: 0.1213888888888889
Review rounds, median: 0.0
